In [12]:
import pkg_resources
import pandas as pd
import numpy as np 
import seaborn as sns
import sklearn
import azure
import matplotlib.pyplot as plt
from azure.ai.ml import MLClient
from azure.ai.ml import command
from azure.identity import DefaultAzureCredential , AzureCliCredential
from azure.core.exceptions import HttpResponseError
import mltable
from sklearn.model_selection import train_test_split
from azure.ai.ml.entities import Data
from azure.ai.ml import automl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input
from azure.ai.ml.entities import Environment
from ucimlrepo import fetch_ucirepo
from azure.identity import AzureCliCredential
from mltable import DataType
from azure.ai.ml.entities import AmlCompute
import mlflow
import subprocess
import yaml
import sys

In [2]:
subprocess.run('azd auth login')

CompletedProcess(args='azd auth login', returncode=0)

In [ ]:
# Connecting to Azure command line interface


credential = AzureCliCredential(subscription= "48f83b14-32a8-4911-b629-ea5aead7fb29")

# Test whether Azure CLI can obtain an Azure management token
token = credential.get_token(
    "https://management.azure.com/.default"
)

print("Authentication successful")

Authentication successful


In [5]:
# connecting to Azure ML studio
ml_client = MLClient(
    credential = credential,
    subscription_id = "48f83b14-32a8-4911-b629-ea5aead7fb29",
    resource_group_name =  "lm1610-rg",
    workspace_name =  "lm_portfolio",
)

In [15]:
cluster = AmlCompute(
    name = 'aml-cluster-v1',
    description = "This compute cluster is created temporarily for running a basic python script",
    size="STANDARD_DS3_V2",            
    min_instances = 0,
    max_instances = 2,
    idle_time_before_scale_down = 120

)

In [16]:
ml_client.begin_create_or_update(cluster)

In [17]:
ml_client.compute.get('aml-cluster-v1')

AmlCompute({'type': 'amlcompute', 'created_on': '2026-11-23T08:14:20.000473+00:00', 'provisioning_state': <ProvisioningState.SUCCEEDED: 'Succeeded'>, 'provisioning_errors': None, 'name': 'aml-cluster-v1', 'description': 'This compute cluster is created temporarily for running a basic python script', 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/computes/aml-cluster-v1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\leste\\OneDrive\\Desktop\\Microsoft VS Code\\Github\\MLOps-Portfolio\\MLOps Notebooks', 'creation_context': None, 'resource_id': None, 'location': 'centralindia', 'size': 'STANDARD_DS3_V2', 'min_instances': 0, 'max_instances': 2, 'idle_time_before_scale_down': 120.0, 'identity': None, 'ssh_public_access_enabled': True, 'ssh_settings': None, 'network_settings': <azure.ai.ml.entities._compute.compute.NetworkSe

In [21]:
for env in ml_client.environments.list():
    print(env.name)

mlops-local-environment
AzureML-AI-Studio-Development
AzureML-ACPT-pytorch-1.13-py38-cuda11.7-gpu
AzureML-ACPT-pytorch-1.12-py38-cuda11.6-gpu
AzureML-ACPT-pytorch-1.12-py39-cuda11.6-gpu
AzureML-ACPT-pytorch-1.11-py38-cuda11.5-gpu
AzureML-ACPT-pytorch-1.11-py38-cuda11.3-gpu
AzureML-responsibleai-0.21-ubuntu20.04-py38-cpu
AzureML-responsibleai-0.20-ubuntu20.04-py38-cpu
AzureML-tensorflow-2.5-ubuntu20.04-py38-cuda11-gpu
AzureML-tensorflow-2.6-ubuntu20.04-py38-cuda11-gpu
AzureML-tensorflow-2.7-ubuntu20.04-py38-cuda11-gpu
AzureML-sklearn-1.0-ubuntu20.04-py38-cpu
AzureML-pytorch-1.10-ubuntu18.04-py38-cuda11-gpu
AzureML-pytorch-1.9-ubuntu18.04-py37-cuda11-gpu
AzureML-pytorch-1.8-ubuntu18.04-py37-cuda11-gpu
AzureML-sklearn-0.24-ubuntu18.04-py37-cpu
AzureML-lightgbm-3.2-ubuntu18.04-py37-cpu
AzureML-pytorch-1.7-ubuntu18.04-py37-cuda11-gpu
AzureML-tensorflow-2.4-ubuntu18.04-py37-cuda11-gpu
AzureML-Triton
AzureML-Designer-Score
AzureML-PyTorch-1.3-CPU
AzureML-VowpalWabbit-8.8.0


In [27]:
for v in ml_client.environments.list(name="mlops-local-environment"):
    print(v.name, v.version, v.image, v.conda_file)

mlops-local-environment 1 mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04 {'dependencies': ['python=3.12.14', 'pip', {'pip': ['pandas == 3.1.0', 'sklearn == 1.9.1', 'numpy == 2.5.4', 'matplotlib.pyplot']}], 'name': 'mlops-local-environment'}


In [28]:
test_job = command(
    code = "C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/MLOps py files/",
    command = "python basic-data-split.py",
    description = "This command job is to configured to run a basic data split",
    environment = "mlops-local-environment@latest",
    display_name = "Basic Data Split job",
    experiment_name = "basic-split-experiment",
    compute = 'aml-cluster-v1'
)

In [31]:
command_job = ml_client.create_or_update(test_job)

In [32]:
command_job.studio_url

'https://ml.azure.com/runs/shy_nut_vhyxmk91pg?wsid=/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourcegroups/lm1610-rg/workspaces/lm_portfolio&tid=38e37b88-a3a1-48cf-9f05-6537427fed24'

In [36]:
ml_client.compute.get('aml-cluster-v1')

AmlCompute({'type': 'amlcompute', 'created_on': '2026-11-23T08:14:20.000473+00:00', 'provisioning_state': <ProvisioningState.SUCCEEDED: 'Succeeded'>, 'provisioning_errors': None, 'name': 'aml-cluster-v1', 'description': 'This compute cluster is created temporarily for running a basic python script', 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/computes/aml-cluster-v1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\leste\\OneDrive\\Desktop\\Microsoft VS Code\\Github\\MLOps-Portfolio\\MLOps Notebooks', 'creation_context': None, 'resource_id': None, 'location': 'centralindia', 'size': 'STANDARD_DS3_V2', 'min_instances': 0, 'max_instances': 2, 'idle_time_before_scale_down': 120.0, 'identity': None, 'ssh_public_access_enabled': True, 'ssh_settings': None, 'network_settings': <azure.ai.ml.entities._compute.compute.NetworkSe

In [ ]:
ml_client.compute.begin_delete('aml-cluster-v1').result()

Invalid type `<class 'type'>` for attribute value. Expected one of bool, str, None, bytes, int, float or a Mapping or Sequence of those types. Value's __str__ method will be called if it exists, otherwise the value will be replaced with None.
